<a href="https://colab.research.google.com/github/Polon-Lubvi/ICTIS/blob/main/01_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Установка

In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets evaluate rouge_score bert_score mlflow pandera gradio sumy datasketch natasha
!pip freeze | grep -iE "^(transformers|peft|trl|bitsandbytes|accelerate|datasets|evaluate|rouge.score|bert.score|mlflow|pandera|gradio|sumy|datasketch|natasha|torch)=="

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
accelerate==1.15.0
bert-score==0.3.13
bitsandbytes==0.50.2
datasets==4.8.5
datasketch==2.0.0
evaluate==0.4.6
gradio==6.31.0
mlflow==3.18.0
natasha==1.6.0
pandera==0.36.0
peft==0.21.2
rouge_score==0.1.2
sumy==0.14.0
torch==2.14.1+cu130
transformers==5.22.1
trl==1.19.0


In [2]:
import os, time, csv, datetime
import numpy as np, pandas as pd, torch
from transformers import set_seed

SEED = 42
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")

GPU: Tesla T4


In [3]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [4]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/tabd26"
for sub in ["data/raw", "data/processed", "checkpoints", "mlruns", "reports"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)

Mounted at /content/drive


In [5]:
!pip uninstall -y -q torchao

## Журнал прогона

In [6]:
LOG = f"{ROOT}/reports/run_log.csv"

def log_step(step, started, rows_in=None, rows_out=None, skipped=False):
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["step", "started", "seconds", "rows_in", "rows_out", "skipped"])
        w.writerow([step, datetime.datetime.fromtimestamp(started).isoformat(timespec="seconds"),
                    round(time.time() - started, 1), rows_in, rows_out, skipped])

## Загрузка данных

In [7]:
from datasets import load_dataset

t0 = time.time()
raw = {}
skipped = True
for split in ["train", "validation", "test"]:
    path = f"{ROOT}/data/raw/gazeta_{split}.parquet"
    if not os.path.exists(path):
        load_dataset("IlyaGusev/gazeta", revision="refs/convert/parquet", split=split).to_parquet(path)
        skipped = False
    raw[split] = pd.read_parquet(path)
    print(split, raw[split].shape)
log_step("load_raw", t0, rows_out=sum(len(d) for d in raw.values()), skipped=skipped)

train (60964, 5)
validation (6369, 5)
test (6793, 5)


## Проверка качества данных

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa

schema = pa.DataFrameSchema(
    {
        "text":    pa.Column(str, [pa.Check.str_length(200, 20000)], nullable=False),
        "summary": pa.Column(str, [pa.Check.str_length(min_value=20)], nullable=False),
        "title":   pa.Column(str, nullable=True),
        "date":    pa.Column(str, nullable=False),
        "url":     pa.Column(str, unique=True, nullable=False),
    },
    checks=[
        pa.Check(lambda df: df["summary"].str.len() < df["text"].str.len(), name="summary_shorter_than_text"),
        pa.Check(lambda df: pd.to_datetime(df["date"], errors="coerce").notna(), name="date_parses"),
    ],
    strict=True,
)

t0 = time.time()
reports = []
for split, df in raw.items():
    try:
        schema.validate(df, lazy=True)
        print(split, "— нарушений нет")
    except pa.errors.SchemaErrors as e:
        fc = e.failure_cases.assign(split=split)
        reports.append(fc)
        print(split, "— нарушений:", len(fc))
quality = pd.concat(reports) if reports else pd.DataFrame()
quality.to_csv(f"{ROOT}/reports/quality.csv", index=False)
log_step("quality", t0, rows_in=sum(len(d) for d in raw.values()), rows_out=len(quality))
quality.groupby(["split", "check"]).size() if len(quality) else "Все правила выполнены"

## Проверка утечки между train и test

In [ ]:
from datasketch import MinHash, MinHashLSH

def shingles(text, k=5):
    w = text.lower().split()
    return {" ".join(w[i:i+k]) for i in range(max(1, len(w) - k + 1))}

def minhash(text):
    m = MinHash(num_perm=64, seed=SEED)
    for s in shingles(text):
        m.update(s.encode("utf-8"))
    return m

t0 = time.time()
train, test = raw["train"], raw["test"]
exact_url  = len(set(train.url) & set(test.url))
exact_text = len(set(train.text) & set(test.text))

lsh = MinHashLSH(threshold=0.8, num_perm=64)
for i, t in enumerate(train.text):
    lsh.insert(str(i), minhash(t))
near = [j for j, t in enumerate(test.text) if lsh.query(minhash(t))]

dates = {s: pd.to_datetime(d.date, errors="coerce") for s, d in raw.items()}
leak = {
    "exact_url": exact_url,
    "exact_text": exact_text,
    "near_duplicates_test": len(near),
    "train_dates": f"{dates['train'].min().date()} — {dates['train'].max().date()}",
    "test_dates":  f"{dates['test'].min().date()} — {dates['test'].max().date()}",
}
pd.Series(leak).to_csv(f"{ROOT}/reports/leakage.csv")
log_step("leakage", t0, rows_in=len(train) + len(test), rows_out=len(near))
leak

## Выборка для оценки

In [8]:
EVAL_PATH = f"{ROOT}/data/processed/eval_sample.parquet"
if not os.path.exists(EVAL_PATH):
    raw["test"].sample(500, random_state=SEED).to_parquet(EVAL_PATH)
eval_df = pd.read_parquet(EVAL_PATH)
eval_df.shape

(500, 5)

## Lead-3 и ROUGE

In [9]:
import re, evaluate
from razdel import sentenize

def lead3(text):
    return " ".join(s.text for s in list(sentenize(text))[:3])

rouge = evaluate.load("rouge")
tokenize = lambda s: re.findall(r"\w+", s.lower())

preds = [lead3(t) for t in eval_df.text]
scores = rouge.compute(predictions=preds, references=list(eval_df.summary), tokenizer=tokenize)
scores

{'rouge1': np.float64(0.25307701155304796),
 'rouge2': np.float64(0.09448342643561934),
 'rougeL': np.float64(0.17591901333312332),
 'rougeLsum': np.float64(0.17529782804928767)}

## Метрики

In [10]:
METRICS = f"{ROOT}/reports/metrics.csv"

def save_result(method, preds, seconds):
    s = rouge.compute(predictions=preds, references=list(eval_df.summary), tokenizer=tokenize)
    row = {"method": method, **{k: round(float(v), 4) for k, v in s.items()},
           "sec_per_doc": round(seconds / len(preds), 3), "n": len(preds)}
    m = pd.read_csv(METRICS) if os.path.exists(METRICS) and os.path.getsize(METRICS) > 1 else pd.DataFrame(columns=["method"])
    m = pd.concat([m[m.method != method], pd.DataFrame([row])], ignore_index=True)
    m.to_csv(METRICS, index=False)
    eval_df.assign(pred=preds).to_parquet(f"{ROOT}/data/processed/preds_{method}.parquet")
    return m

t0 = time.time()
save_result("lead3", [lead3(t) for t in eval_df.text], time.time() - t0)

,method,rouge1,rouge2,rougeL,rougeLsum,sec_per_doc,n
0,textrank,0.1998,0.0497,0.1330,0.1328,0.007,500.0
1,zero_shot,0.2316,0.0696,0.1586,0.1587,1.670,500.0
2,qlora_r16_attn,0.2311,0.0760,0.1650,0.1647,1.536,500.0
3,qlora_r16_all,0.2314,0.0757,0.1655,0.1654,1.471,500.0
4,lead3,0.2531,0.0945,0.1759,0.1753,0.001,500.0


## TextRank

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def textrank(text, n=3):
    sents = [s.text for s in sentenize(text)]
    if len(sents) <= n:
        return " ".join(sents)
    sim = cosine_similarity(TfidfVectorizer(token_pattern=r"\w+").fit_transform(sents))
    np.fill_diagonal(sim, 0)
    rank = nx.pagerank(nx.from_numpy_array(sim))
    top = sorted(sorted(rank, key=rank.get, reverse=True)[:n])
    return " ".join(sents[i] for i in top)

t0 = time.time()
save_result("textrank", [textrank(t) for t in eval_df.text], time.time() - t0)
log_step("baselines_extractive", t0, rows_in=len(eval_df))

## Zero-shot Qwen

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
#model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
PROMPT = "Кратко перескажи новость в 1–3 предложениях.\n\n{text}"

def build(text):
    msgs = [{"role": "user", "content": PROMPT.format(text=text[:6000])}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(texts, bs=8, max_new=160):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok([build(t) for t in texts[i:i+bs]], return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
        out += tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in out]

#t0 = time.time()
#print(save_result("zero_shot", generate(list(eval_df.text)), time.time() - t0))
#log_step("baseline_zero_shot", t0, rows_in=len(eval_df))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Данные для обучения

In [12]:
import mlflow, shutil
from datasets import Dataset
from transformers import BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

globals().pop("model", None); torch.cuda.empty_cache()

MLFLOW_DB = "/content/mlflow.db"
if os.path.exists(f"{ROOT}/mlruns/mlflow.db") and not os.path.exists(MLFLOW_DB):
    shutil.copy(f"{ROOT}/mlruns/mlflow.db", MLFLOW_DB)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{MLFLOW_DB}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "gazeta_lora"

tr_df = raw["train"]
tr_df = tr_df[tr_df.text.str.len().between(200, 20000) & (tr_df.summary.str.len() < tr_df.text.str.len())]

def to_chat(df):
    return Dataset.from_list([
        {"prompt": [{"role": "user", "content": PROMPT.format(text=t[:3000])}],
         "completion": [{"role": "assistant", "content": s}]}
        for t, s in zip(df.text, df.summary)])

train_ds = to_chat(tr_df.sample(2000, random_state=SEED))
val_ds = to_chat(raw["validation"].sample(200, random_state=SEED))

## Обучение LoRA

In [14]:
ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
CONFIGS = {
    "qlora_r8_attn":  dict(r=8,  target=ATTN, quant=True),
    "qlora_r16_attn": dict(r=16, target=ATTN, quant=True),
    "qlora_r16_all":  dict(r=16, target="all-linear", quant=True),
    "lora_r16_attn":  dict(r=16, target=ATTN, quant=False),
}

def train(name):
    mlflow.end_run()
    c, out = CONFIGS[name], f"{ROOT}/checkpoints/{name}"
    if os.path.exists(f"{out}/final/adapter_config.json"):
        return print(name, "уже обучена")
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16) if c["quant"] else None
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16,
                                                quantization_config=quant, device_map="cuda")
    args = SFTConfig(output_dir=out, run_name=name, num_train_epochs=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,
        fp16=True, gradient_checkpointing=True, max_length=1536,
        logging_steps=5, eval_strategy="steps", eval_steps=25,
        save_steps=25, save_total_limit=1, report_to="mlflow", seed=SEED)
    lora = LoraConfig(r=c["r"], lora_alpha=2 * c["r"], lora_dropout=0.05,
                      target_modules=c["target"], task_type="CAUSAL_LM")
    tr = SFTTrainer(model=base, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    processing_class=tok, peft_config=lora)
    for p in tr.model.parameters():
        if p.requires_grad: p.data = p.data.float()
    print("примеров в обучении:", len(tr.train_dataset))
    tr.train(resume_from_checkpoint=get_last_checkpoint(out) if os.path.isdir(out) else None)
    tr.save_model(f"{out}/final")
    print(name, f"{(time.time() - t0) / 60:.1f} мин,",
          f"пик памяти GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ")
    shutil.copy(MLFLOW_DB, f"{ROOT}/mlruns/mlflow.db")
    log_step(f"train_{name}", t0, rows_in=len(train_ds))
    del tr, base; torch.cuda.empty_cache()

train("qlora_r8_attn")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

... (5 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

примеров в обучении: 2000


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,1.283054,1.155003,1.206929,0.730200,472086.000000
50,1.309188,1.148054,1.204162,0.731523,943994.000000
75,1.170625,1.136862,1.162440,0.734928,1414534.000000
100,1.234266,1.134518,1.151989,0.735460,1889057.000000
125,1.284709,1.134061,1.155678,0.735413,2365316.000000


qlora_r8_attn 30.0 мин, пик памяти GPU 1.9 ГБ


## Оценка LoRA

In [15]:
from peft import PeftModel

def eval_lora(name):
    global model
    mlflow.end_run()
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, f"{ROOT}/checkpoints/{name}/final").merge_and_unload().eval()
    t0 = time.time()
    m = save_result(name, generate(list(eval_df.text)), time.time() - t0)
    row = m[m.method == name].iloc[0]
    run = mlflow.search_runs(experiment_names=["gazeta_lora"],
                             filter_string=f"attributes.run_name = '{name}'",
                             order_by=["start_time DESC"]).iloc[0]
    with mlflow.start_run(run_id=run.run_id):
        mlflow.log_metrics({f"eval_{k}": float(row[k]) for k in ["rouge1", "rouge2", "rougeL", "sec_per_doc"]})
    shutil.copy(MLFLOW_DB, f"{ROOT}/mlruns/mlflow.db")
    log_step(f"eval_{name}", t0, rows_in=len(eval_df))
    del model, base; torch.cuda.empty_cache()
    return m

eval_lora("qlora_r8_attn")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

,method,rouge1,rouge2,rougeL,rougeLsum,sec_per_doc,n
0,textrank,0.1998,0.0497,0.1330,0.1328,0.007,500.0
1,zero_shot,0.2316,0.0696,0.1586,0.1587,1.670,500.0
2,qlora_r16_attn,0.2311,0.0760,0.1650,0.1647,1.536,500.0
3,qlora_r16_all,0.2314,0.0757,0.1655,0.1654,1.471,500.0
4,lead3,0.2531,0.0945,0.1759,0.1753,0.001,500.0
5,qlora_r8_attn,0.2297,0.0751,0.1658,0.1658,1.394,500.0


## Доверительные интервалы

In [ ]:
METHODS = ["lead3", "textrank", "zero_shot", "qlora_r16_attn", "qlora_r16_all"]
idx = np.random.default_rng(SEED).integers(0, len(eval_df), size=(1000, len(eval_df)))

per_doc = {}
for m in METHODS:
    p = pd.read_parquet(f"{ROOT}/data/processed/preds_{m}.parquet")
    per_doc[m] = rouge.compute(predictions=list(p.pred), references=list(p.summary),
                               tokenizer=tokenize, use_aggregator=False)

rows = []
for m in METHODS:
    for k in ["rouge1", "rouge2", "rougeL"]:
        v, base = np.array(per_doc[m][k]), np.array(per_doc["zero_shot"][k])
        boot, diff = v[idx].mean(1), (v - base)[idx].mean(1)
        rows.append({"method": m, "metric": k, "mean": round(v.mean(), 4),
                     "ci_low": round(np.percentile(boot, 2.5), 4),
                     "ci_high": round(np.percentile(boot, 97.5), 4),
                     "p_better_zero_shot": round((diff > 0).mean(), 3)})
ci = pd.DataFrame(rows)
ci.to_csv(f"{ROOT}/reports/bootstrap_ci.csv", index=False)
ci.pivot(index="method", columns="metric", values=["mean", "ci_low", "ci_high", "p_better_zero_shot"])

mean                  ci_low                 ci_high  \
metric          rouge1  rouge2  rougeL  rouge1  rouge2  rougeL  rouge1   
method                                                                   
lead3           0.2530  0.0946  0.1755  0.2450  0.0879  0.1687  0.2616   
qlora_r16_all   0.2311  0.0758  0.1655  0.2226  0.0696  0.1591  0.2396   
qlora_r16_attn  0.2309  0.0759  0.1650  0.2228  0.0694  0.1585  0.2383   
textrank        0.1998  0.0497  0.1329  0.1932  0.0448  0.1275  0.2065   
zero_shot       0.2314  0.0696  0.1585  0.2240  0.0645  0.1528  0.2389   

                               p_better_zero_shot                
metric          rouge2  rougeL             rouge1 rouge2 rougeL  
method                                                           
lead3           0.1014  0.1825              1.000  1.000  1.000  
qlora_r16_all   0.0824  0.1728              0.491  0.981  0.975  
qlora_r16_attn  0.0820  0.1718              0.483  0.987  0.974  
textrank        0.0549  0.1383              0.000  0.000  0.000  
zero_shot       0.0749  0.1644              0.000  0.000  0.000

## Галлюцинации

In [ ]:
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, NewsMorphTagger, MorphVocab, Doc
seg, emb, voc = Segmenter(), NewsEmbedding(), MorphVocab()
ner, morph = NewsNERTagger(emb), NewsMorphTagger(emb)

def entities(text):
    d = Doc(text); d.segment(seg); d.tag_morph(morph); d.tag_ner(ner)
    for s in d.spans: s.normalize(voc)
    return {(s.type, s.normal.lower()) for s in d.spans}

t0 = time.time()
src = [entities(t) for t in eval_df.text]
found, summary = [], []
for m in METHODS:
    p = pd.read_parquet(f"{ROOT}/data/processed/preds_{m}.parquet")
    n_bad = 0
    for i, (pred, text) in enumerate(zip(p.pred, p.text)):
        bad = [e for e in entities(pred) if e not in src[i] and e[1] not in text.lower()]
        n_bad += bool(bad)
        found += [{"method": m, "doc": i, "type": t, "entity": e} for t, e in bad]
    summary.append({"method": m, "share_with_halluc": round(n_bad / len(p), 3),
                    "entities_per_doc": round(sum(f["method"] == m for f in found) / len(p), 3)})
pd.DataFrame(found).to_csv(f"{ROOT}/reports/hallucinations.csv", index=False)
h = pd.DataFrame(summary); h.to_csv(f"{ROOT}/reports/hallucinations_summary.csv", index=False)
log_step("hallucinations_ner", t0, rows_in=len(eval_df) * len(METHODS))
h

/usr/local/lib/python3.13/dist-packages/pymorphy2/analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


,method,share_with_halluc,entities_per_doc
0,lead3,0.000,0.000
1,textrank,0.000,0.000
2,zero_shot,0.228,0.300
3,qlora_r16_attn,0.200,0.268
4,qlora_r16_all,0.200,0.274


## Ручная оценка

In [ ]:
docs = np.random.default_rng(SEED).choice(len(eval_df), 50, replace=False)
items = []
for m in ["zero_shot", "qlora_r16_all"]:
    p = pd.read_parquet(f"{ROOT}/data/processed/preds_{m}.parquet")
    items += [{"doc": int(d), "method": m, "text": p.text.iloc[d][:4000], "summary": p.pred.iloc[d]} for d in docs]
me = pd.DataFrame(items).sample(frac=1, random_state=SEED).reset_index(drop=True)
me.insert(0, "item", range(1, len(me) + 1))
me[["item", "doc", "method"]].to_csv(f"{ROOT}/reports/manual_eval_key.csv", index=False)
sheet = me[["item", "text", "summary"]].assign(полнота="", точность="", связность="", галлюцинация="")
for r in [1, 2]:
    sheet.to_excel(f"{ROOT}/reports/manual_eval_rater{r}.xlsx", index=False)

## Тональность: данные и базовые модели

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import f1_score

URL = "https://raw.githubusercontent.com/dialogue-evaluation/RuSentNE-evaluation/main"
sent = {}
for split, f in [("train", "train_data.csv"), ("test", "validation_data_labeled.csv")]:
    path = f"{ROOT}/data/raw/rusentne_{split}.csv"
    if not os.path.exists(path):
        pd.read_csv(f"{URL}/{f}", sep="\t").to_csv(path, index=False)
    sent[split] = pd.read_csv(path)
    print(split, sent[split].shape, sent[split].label.value_counts().to_dict())

def mark(r):
    s, a, b = r.sentence, r.entity_pos_start_rel, r.entity_pos_end_rel
    return s[:a] + " [E] " + s[a:b] + " [/E] " + s[b:]

for d in sent.values():
    d["x"] = d.apply(mark, axis=1)

SENT_METRICS = f"{ROOT}/reports/sentiment_metrics.csv"
def sent_result(method, pred, seconds):
    y, org = sent["test"].label, sent["test"].entity_tag == "ORGANIZATION"
    row = {"method": method,
           "macro_f1": round(f1_score(y, pred, average="macro"), 4),
           "f1_pn": round(f1_score(y, pred, labels=[-1, 1], average="macro"), 4),
           "macro_f1_org": round(f1_score(y[org], pred[org], average="macro"), 4),
           "sec_per_doc": round(seconds / len(y), 5)}
    m = pd.read_csv(SENT_METRICS) if os.path.exists(SENT_METRICS) else pd.DataFrame(columns=["method"])
    m = pd.concat([m[m.method != method], pd.DataFrame([row])], ignore_index=True)
    m.to_csv(SENT_METRICS, index=False)
    return m

t0 = time.time()
clf = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
    LogisticRegression(max_iter=2000, class_weight="balanced", C=2))
clf.fit(sent["train"].x, sent["train"].label)
t1 = time.time()
sent_result("tfidf_logreg", clf.predict(sent["test"].x), time.time() - t1)
log_step("sentiment_tfidf", t0, rows_in=len(sent["train"]), rows_out=len(sent["test"]))

train (6637, 6) {0: 4774, -1: 1007, 1: 856}
test (2845, 6) {0: 2045, -1: 438, 1: 362}


In [ ]:
from transformers import pipeline

t0 = time.time()
tiny = pipeline("text-classification", model="cointegrated/rubert-tiny-sentiment-balanced", device=-1)
LABEL = {"negative": -1, "neutral": 0, "positive": 1}
out = tiny(list(sent["test"].sentence), batch_size=32, truncation=True)
pred = np.array([LABEL[o["label"]] for o in out])
print(sent_result("rubert_tiny", pred, time.time() - t0))
log_step("sentiment_rubert_tiny", t0, rows_in=len(sent["test"]))

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 47.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/377 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/241k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/468k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

         method  macro_f1   f1_pn  macro_f1_org  sec_per_doc
0  tfidf_logreg    0.5333  0.4171        0.5510      0.00050
1   rubert_tiny    0.3607  0.1422        0.3766      0.01897


## Мониторинг брендов

### Мониторинг брендов: поиск упоминаний

In [ ]:
import plotly.express as px

t0 = time.time()
news = pd.read_parquet(f"{ROOT}/data/raw/gazeta_test.parquet").sample(2000, random_state=SEED)
rows = []
for date, text in zip(news.date, news.text):
    d = Doc(text); d.segment(seg); d.tag_morph(morph); d.tag_ner(ner)
    for sp in d.spans:
        if sp.type != "ORG":
            continue
        sp.normalize(voc)
        s = next((s for s in d.sents if s.start <= sp.start < s.stop), None)
        if s is None:
            continue
        a, b = sp.start - s.start, sp.stop - s.start
        rows.append({"month": date[:7], "brand": sp.normal,
                     "x": s.text[:a] + " [E] " + s.text[a:b] + " [/E] " + s.text[b:]})
bm = pd.DataFrame(rows)
bm["label"] = clf.predict(bm.x)

24175 упоминаний, 6335 организаций


### Мониторинг брендов: дашборд

In [ ]:
import plotly.express as px

MEDIA = {"газета.ru", "сми", "тасс", "риа «новости»", "интерфакс", "рбк", "reuters",
         "bloomberg", "известия", "коммерсант", "ведомости", "bbc", "cnn"}
bm = bm[~bm.brand.str.lower().isin(MEDIA)]

agg = bm.groupby("brand").label.agg(mentions="size",
                                    positive=lambda v: (v == 1).mean(),
                                    negative=lambda v: (v == -1).mean()).reset_index()
agg["net"] = (agg.positive - agg.negative).round(3)
agg.sort_values("mentions", ascending=False).to_csv(f"{ROOT}/reports/brand_sentiment.csv", index=False)
top = agg.nlargest(15, "mentions")

fig1 = px.bar(top.sort_values("mentions"), x="mentions", y="brand", color="net", orientation="h",
              color_continuous_scale="RdYlGn", range_color=[-0.5, 0.5],
              title="Упоминания и тональность: топ-15 организаций")
monthly = (bm[bm.brand.isin(top.brand.head(5))].groupby(["month", "brand"]).label
           .agg(net=lambda v: (v == 1).mean() - (v == -1).mean()).reset_index())
fig2 = px.line(monthly, x="month", y="net", color="brand", markers=True,
               title="Чистая тональность по месяцам: топ-5 организаций")
with open(f"{ROOT}/reports/brand_dashboard.html", "w", encoding="utf-8") as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs="cdn"))
    f.write(fig2.to_html(full_html=False, include_plotlyjs=False))
log_step("brand_monitoring", t0, rows_in=len(news), rows_out=len(bm))
print(len(bm), "упоминаний,", bm.brand.nunique(), "организаций")
fig1.show(); fig2.show()

21721 упоминаний, 6322 организаций
